In [1]:
import re
import pandas as pd
import json
import os
os.chdir('..')

In [2]:
from transformer_lens import HookedTransformer, HookedTransformerConfig
import pickle
import torch
os.environ['CUDA_VISIBLE_DEVICES'] = '7'

/raid/home/m13521157/absa-eap-ig/enveap/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Automatically select device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
def load_finetuned_model_lens_from_dir(dir: str, device: str = device) -> HookedTransformer:
    """
    Load a fine-tuned TransformerLens model from a specified directory.

    Args:
        dir (str): Directory containing the model files.
        device (str): Device to load the model onto.
    
    Returns:
        HookedTransformer: The loaded TransformerLens model.
    """
    with open(os.path.join(dir, 'model_config.pkl'), 'rb') as f:
        new_cfg_dict = pickle.load(f)
    new_cfg = HookedTransformerConfig.from_dict(new_cfg_dict)
    new_model = HookedTransformer(new_cfg)
    new_model.load_state_dict(torch.load(os.path.join(dir, 'model.pt'), map_location=device))
    return new_model

In [5]:
model = load_finetuned_model_lens_from_dir('outputs/modelsadamw/eap/corrected_splitopinion_typocorrected/circuit-indo_finetune-indo/seed_123/aos_sequence_variants/full_sft/2025-10-11 20:42:22.646774_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20')

In [9]:
text = "kamar saya ada kendala di ac tidak berfungsi optimal . dan juga wifi koneksi kurang stabil . [A] [O] [S] =>"
target = "[A] ac [O] tidak berfungsi optimal [S] negative [SSEP] [A] wifi koneksi [O] kurang stabil [S] negative"
check_text = " [A] [O] [S] => [A] ac [O] tidak berfungsi optimal [S] negative [SSEP] [A] wifi koneksi [O] kurang stabil [S] negative"

In [12]:
print(model.to_str_tokens(text)[-10:])

[' [', 'A', ']', ' [', 'O', ']', ' [', 'S', ']', ' =>']


In [14]:
model.to_str_tokens(check_text)

[' [',
 'A',
 ']',
 ' [',
 'O',
 ']',
 ' [',
 'S',
 ']',
 ' =>',
 ' [',
 'A',
 ']',
 ' ac',
 ' [',
 'O',
 ']',
 ' tidak',
 ' ber',
 'fung',
 'si',
 ' optimal',
 ' [',
 'S',
 ']',
 ' negative',
 ' [',
 'S',
 'SEP',
 ']',
 ' [',
 'A',
 ']',
 ' wifi',
 ' k',
 'oneksi',
 ' [',
 'O',
 ']',
 ' kur',
 'ang',
 ' stabil',
 ' [',
 'S',
 ']',
 ' negative']

In [ ]:
# Define your special tokens based on the example
special_tokens = ["[A]", "[O]", "[S]", "[SSEP]", "null", "positive", "negative", "neutral"]

# Add tokens to the tokenizer
for token in special_tokens:
    model.tokenizer.add_tokens(token)

# Convenience function to get token IDs
def get_token_ids(tokens):
    return torch.tensor(model.tokenizer.convert_tokens_to_ids(tokens), device=model.cfg.device)

In [ ]:
# --- Define the token sets for our constraints ---

# 1. Tokens from the input sentence
input_sentence = "hotel bersih , walaupun hotel lama , dan dari luar dekorasinya agak tua dan creepy , tetapi oke kok pas m ."
# Tokenize the input sentence and get unique token IDs
input_word_ids = torch.unique(get_token_ids(model.tokenizer.tokenize(input_sentence)))

# 2. Sentiment tokens
sentiment_word_ids = get_token_ids(["positive", "negative", "neutral", "great", "bad"])

# 3. Special control tokens
ssep_token_id = get_token_ids(["[SSEP]"])
all_control_token_ids = get_token_ids(["[A]", "[O]", "[S]", "[SSEP]"])

# 4. The 'null' token for implicit aspects
null_token_id = get_token_ids(["null"])

# Combine input words and null token for [A] and [O] states
allowed_ao_content_ids = torch.cat([input_word_ids, null_token_id])

print(f"Found {len(input_word_ids)} unique input token IDs.")
print(f"Sentiment token IDs: {sentiment_word_ids.tolist()}")

In [ ]:
def constrained_generate(model, prompt, max_new_tokens=50):
    """
    Generates text token by token, applying constraints to the logits at each step.
    """
    # Tokenize the initial prompt
    generated_tokens = get_token_ids(model.tokenizer.tokenize(prompt)).unsqueeze(0)
    
    # --- Generation Loop ---
    for i in range(max_new_tokens):
        # Get the logits from the model
        logits = model(generated_tokens)
        
        # We only care about the logits for the very next token
        next_token_logits = logits[0, -1, :]
        
        # --- Determine Current State and Allowed Tokens ---
        current_state = None
        # Find the last special token to determine the state
        for token_id in reversed(generated_tokens[0]):
            if token_id in all_control_token_ids:
                current_state = model.tokenizer.decode(token_id)
                break
        
        # Default to allowing control tokens if no state is found yet
        if current_state is None:
             allowed_token_ids = get_token_ids(["[A]"]) # Start with [A]
        elif current_state == "[A]":
            # Allow input words, null, or the next state markers [O] or [SSEP]
            allowed_token_ids = torch.cat([allowed_ao_content_ids, get_token_ids(["[O]", "[SSEP]"])])
        elif current_state == "[O]":
            # Allow input words, null, or the next state markers [S] or [SSEP]
            allowed_token_ids = torch.cat([allowed_ao_content_ids, get_token_ids(["[S]", "[SSEP]"])])
        elif current_state == "[S]":
            # Allow sentiment words or [SSEP] to end the triplet
            allowed_token_ids = torch.cat([sentiment_word_ids, ssep_token_id])
        elif current_state == "[SSEP]":
            # After a separator, start a new triplet with [A]
            allowed_token_ids = get_token_ids(["[A]"])
        
        # --- Create and Apply Logit Mask ---
        # Create a mask that is -inf for all tokens except the allowed ones
        mask = torch.full_like(next_token_logits, -float('Inf'))
        mask[allowed_token_ids] = 0
        
        # Apply the mask
        masked_logits = next_token_logits + mask
        
        # --- Sample the Next Token ---
        # Use argmax for greedy decoding (always pick the most likely token)
        next_token = torch.argmax(masked_logits).unsqueeze(0)
        
        # Append the new token to our sequence
        generated_tokens = torch.cat([generated_tokens, next_token.unsqueeze(0)], dim=1)
        
        # Stop if we generate the end-of-sequence token
        if next_token == model.tokenizer.eos_token_id:
            break
            
    return model.tokenizer.decode(generated_tokens[0])